In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

In [2]:
from test_model import test
from argparse import Namespace
import glob
import numpy as np
from concurrent import futures
import multiprocessing
import copy
import json

c:\Users\Administrator\.conda\envs\PyTorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
args = {
    'data_path': '../Data/',
    'pretrained_model': 'incep', #supcon', #'resnet',
    'pretrained_path': '',
    'window_size': 29,
    'strided': 15,
    'batch_size': 512,
    'num_workers': 8,
    'trial_id': 1,
    'ckpt' : 50,
    'car_model': None
}

args = Namespace(**args)

In [4]:
bs = 512
pretrained_paths = sorted(glob.glob(f'../save/inception*'))
# pretrained_paths = sorted(glob.glob(f'../save/SupCon.resnet18*_bs{bs}*'))
# pretrained_paths = sorted(glob.glob('../save/smallresnet18.ce?_*_lr0.001_*'))

pretrained_paths

[]

In [5]:
num_cpu = multiprocessing.cpu_count()
workers = max(num_cpu, len(pretrained_paths))

with futures.ProcessPoolExecutor(workers) as executor:
    to_do = []
    for path in pretrained_paths:
        args.trial_id = path.split('_')[0][-1]
        args.pretrained_path = f'{path}/models/'
        future = executor.submit(test, copy.deepcopy(args), 
                                 {'verbose': False, 'is_cuda': True})
        to_do.append(future)
        
    
    total_results = {}
    for future in futures.as_completed(to_do):
        try:
            results = future.result()
            for k, v in results.items():
                total_results.setdefault(k, [])
                total_results[k].append(v)
        except Exception as error:
            print('An exception occurred: {}'.format(error))

In [6]:
print('Final results')
total_results = {k: np.stack(v, axis=0) for k, v in total_results.items()}
for k, v in total_results.items():
    print(k, list("{0:0.4f}".format(i) for i in v.mean(axis=0)))

Final results


In [7]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

In [8]:
file = f'{args.pretrained_model}{bs}.json'
with open(f'../reports/{file}', 'w') as convert_file:
     convert_file.write(json.dumps(total_results, cls=NumpyEncoder))